# 🛍️ CustomerIQ - Day 1: E-Commerce Exploratory Data Analysis & RFM Segmentation

**Objective:**
1. Load 10,000+ transactional records from modern e-commerce platform.
2. Perform deep exploratory data analysis (AOV, GMV, fulfillment delays, complaints).
3. Engineer RFM (Recency, Frequency, Monetary) metrics for each distinct customer.
4. Segment customers into cohorts (Champions, Loyal, At-Risk, Lost).
5. Define Churn ground truth for Predictive Modeling (Day 2).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Segoe UI'

# 1. Load Transactions Dataset
df = pd.read_csv('../data/ecommerce_transactions_raw.csv')
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
print(f"Total Transactions Loaded: {len(df):,}")
df.head()

In [ ]:
# 2. Key Business Metrics Overview
total_gmv = df['Order_Value_INR'].sum()
aov = df['Order_Value_INR'].mean()
unique_cust = df['Customer_ID'].nunique()

print(f"Total Gross Merchandise Value (GMV): INR {total_gmv:,.2f}")
print(f"Average Order Value (AOV): INR {aov:,.2f}")
print(f"Unique Customers: {unique_cust:,}")

In [ ]:
# 3. Revenue by Product Category
cat_perf = df.groupby('Product_Category').agg(
    Revenue=('Order_Value_INR', 'sum'),
    Orders=('Transaction_ID', 'count'),
    Avg_Ticket=('Order_Value_INR', 'mean')
).sort_values(by='Revenue', ascending=False)
cat_perf

In [ ]:
# 4. Customer-Level Aggregation for RFM
snapshot_date = df['Order_Date'].max() + pd.Timedelta(days=1)

customer_df = df.groupby('Customer_ID').agg(
    Customer_Age=('Customer_Age', 'first'),
    Customer_Gender=('Customer_Gender', 'first'),
    City=('City', 'first'),
    City_Tier=('City_Tier', 'first'),
    Customer_Tenure_Months=('Customer_Tenure_Months', 'first'),
    Total_Orders=('Transaction_ID', 'count'),
    Total_Spend=('Order_Value_INR', 'sum'),
    Avg_Order_Value=('Order_Value_INR', 'mean'),
    Last_Order_Date=('Order_Date', 'max'),
    Total_Delivery_Delay_Days=('Delivery_Delay_Days', 'sum'),
    Total_Complaints=('Complaints_Filed', 'sum'),
    Avg_Satisfaction_Score=('Satisfaction_Score', 'mean')
).reset_index()

# Compute Recency in Days
customer_df['Recency_Days'] = (snapshot_date - customer_df['Last_Order_Date']).dt.days
customer_df.head()

In [ ]:
# 5. RFM Scoring (1 to 5 Quantiles)
customer_df['R_Score'] = pd.qcut(customer_df['Recency_Days'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
customer_df['F_Score'] = pd.qcut(customer_df['Total_Orders'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
customer_df['M_Score'] = pd.qcut(customer_df['Total_Spend'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_customer(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions (VIP)'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Promising Newcomers'
    elif r <= 2 and f >= 3:
        return 'At-Risk (High Spenders Dropping)'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost / Inactive'
    else:
        return 'Potential Regulars'

customer_df['Customer_Segment'] = customer_df.apply(segment_customer, axis=1)
customer_df['Customer_Segment'].value_counts()

In [ ]:
# 6. Inspect Saved Processed Dataset
processed_df = pd.read_csv('../data/customer_intelligence_processed.csv')
print(f"Processed Data Rows: {len(processed_df):,}, Columns: {processed_df.shape[1]}")
print(f"Overall Churn Rate: {processed_df['Is_Churned'].mean() * 100:.2f}%")
processed_df[['Customer_ID', 'Customer_Segment', 'Recency_Days', 'Total_Spend', 'Is_Churned']].head(10)